# Filter Audit Viz

Loads `output/eval/` data, builds a `{value: color}` map and renders graphs:

| Audit | Metric | Visualization | Description |
| :--- | :--- | :--- | :--- |
| 1 | CRR | heatmap, scatter | cumulative rejection rate |
| 1 |  NPMI | bar graphs | normalized pointwise mutual information |
| 1 | PRR | Line series | progressive rejection rate at each stage |
| 1, 2 | JSD | table | Jensen-Shannon divergence |

In [1]:
import os
import sys

# 1. Calculate the absolute path to the parent directory
root_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

# 2. Add the parent directory to sys.path if it isn't already there
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

In [2]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from preprocess.manifest import PreprocessManifest
from utils import load_config

## Load data

In [3]:
OUTPUT_DIR = Path("../output/eval")
PMI_PATH = OUTPUT_DIR / "pmi"
SANKEY_PATH = OUTPUT_DIR / "retention/sankeys.json"
PLOT_PATH = OUTPUT_DIR / "retention/plot_data"
CONFIG_PATH = "../config/preprocess/enrich.json"

preprocess_manifest = PreprocessManifest("../output/preprocess/manifest.json")
ENRICH_MAP = load_config(CONFIG_PATH)
pmi_data = {
    p.stem: pd.read_csv(p)
    for p in sorted(PMI_PATH.glob("*.csv"))
}

with open(SANKEY_PATH) as f:
    sankeys = json.load(f)

series = {
    p.stem.replace("retention_", ""): pd.read_csv(p, dtype={"value": str})
    for p in sorted(PLOT_PATH.glob("retention_*.csv"))
}

# crr_or_data = crr_or_data.loc[crr_or_data.cluster != 5]
b_ors = pd.read_csv("../b_ors.csv")

## Color mapping

Metrics are color coded for 3 cases:
1. Rows are aggregated on a single feature value -> feature value maps directly to a color
2. Rows are aggregated on feature combinations -> feature combo resolves to primary feature
3. Rows are aggregated on component values -> component values are resolved to cluster IDs

In [4]:
PALETTE = px.colors.qualitative.Set3
EXCLUDED_COLOR = "rgba(150,150,150,0.55)" # Used for Sankey diagrams (abandoned)

def resolve_cluster_for_component(component_val: str, enrich_map=ENRICH_MAP) -> str:
    for source, data in enrich_map.items():
        components = data.get("components", {})
        default = data.get("default", None)
        if default:
            return default
        if component_val in components:
            return components[component_val]

def resolve_type_for_genre(genre_val: str) -> str:
    if genre_val.startswith("W"):
        return "W"
    if genre_val.startswith("S"):
        return "S"
    return genre_val
    
def get_color_key(parts, spec_feature_names, enrich_map=ENRICH_MAP["distribution"]["corpora"]):
    if not spec_feature_names:
        return "overall"
    elif spec_feature_names[0:3] == ["source", "genre", "cluster"]:
        return parts[-1]
    elif spec_feature_names[0:3] == ["source", "genre", "component"]:
        return resolve_cluster_for_component(parts[-1], enrich_map)
    elif spec_feature_names[0:2] == ["mobility", "component"]:
        return resolve_cluster_for_component(parts[-1], enrich_map)
    
    primary_feature = spec_feature_names[0]
    val = parts[0]

    if primary_feature == "component":
        return resolve_cluster_for_component(val, enrich_map)
    if primary_feature == "genre":
        return resolve_type_for_genre(val)
    return val 
        
all_values = set()
processed_series = {}

for key, df in series.items():
    if "value" not in df.columns:
        continue
    df["value_parts"] = df.value.str.split(" / ")
    spec_feature_names = key.replace("by_", "").replace("v2", "").replace("v1", "").replace("_in", "").split("_")
    df["color_key"] = df["value_parts"].apply(lambda parts: get_color_key(parts, spec_feature_names))

    df.sort_values(by="value", inplace=True)
    all_values |= set(df.color_key.unique())
    # processed_series[key] = df
    
all_values.discard("total")
all_values = sorted(list(all_values))

COLOR_MAP = {v: PALETTE[i % len(PALETTE)] for i, v in enumerate(all_values)}

print(f"{len(COLOR_MAP)} values colored")

15 values colored


In [5]:
COLOR_MAP

{'1': 'rgb(141,211,199)',
 '2': 'rgb(255,255,179)',
 '3': 'rgb(190,186,218)',
 '4': 'rgb(251,128,114)',
 '5': 'rgb(128,177,211)',
 'S': 'rgb(253,180,98)',
 'W': 'rgb(179,222,105)',
 'codeswitch': 'rgb(252,205,229)',
 'glowbe': 'rgb(217,217,217)',
 'high': 'rgb(188,128,189)',
 'ice': 'rgb(204,235,197)',
 'lince': 'rgb(255,237,111)',
 'low': 'rgb(141,211,199)',
 'medium': 'rgb(255,255,179)',
 'web': 'rgb(190,186,218)'}

In [6]:
for key, df in series.items():
    if not key.endswith("local") and key != "overall":
        pdf = pmi_data[key]
        pdf["feature_value"] = pdf["feature_value"].astype(str)
        pdf["value_parts"] = pdf.feature_value.str.split(" / ")
        spec_feature_names = key.replace("by_", "").replace("_in", "").split("_")
        pdf["color_key"] = pdf["value_parts"].apply(lambda parts: get_color_key(parts, spec_feature_names))

b_ors["level"] = b_ors["level"].astype(str)
b_ors["value_parts"] = b_ors.level.str.split("/")
spec_feature_names = ["mobility"]
b_ors["color_key"] = b_ors["value_parts"].apply(lambda parts: get_color_key(parts, spec_feature_names))

# pivot_df = df.pivot(index="cluster", columns="level", values="rejection_p_hat")

In [7]:
f_level_map = {
    "low": 1,
    "medium": 2,
    "high": 3,
}
country_order = [
    "us",
    "gb",  # Cluster 1
    "ca",
    "ie",
    "au",
    "nz",  # Cluster 2
    "in",
    "sg",
    "ph",
    "hk",  # Cluster 3
    "ng",
    "ke",
    "tz",
    "jm",  # Cluster 4
]

f_order = [
    "unknown (web)",
    "low",
    "medium",
    "high"
]

# Filter Reason Breakdowns

In [8]:
per_doc_data = pd.read_csv(
    "../fineweb_output.csv", 
    dtype={
        "doc_id": "string",
        "language": "string",
        "language_score": "Float64",
        "filter_reason": "string",
        "result": "string",
        "stage_excluded": "Int64",
        "source": "string",
        "genre": "string",
        "component": "string",
        "cluster_id": "string",
        "mobility_v1": "string",
        "mobility_v2": "string"
    }
    )
per_doc_data.loc[per_doc_data.source=="glowbe", "mobility_v2"] = "unknown (web)"

In [9]:
agg = per_doc_data.groupby(['mobility_v2', 'cluster_id']).size().reset_index(name='count')
agg['log_count'] = np.log10(agg['count'] + 1)

fig = px.scatter(
    agg,
    x='mobility_v2',
    y='cluster_id',
    size='log_count',
    color="count",
    size_max=30,
    color_continuous_scale='Viridis',
    labels={
        "mobility_v2": "Register level",
        "filter_reason": "Filter reason",
        "cluster_id": "Provenance group",
        # "component": "Provenance (country)",
        "proportion": "Proportion"
    }
)
fig.update_layout(width=500, height=500)
# fig.show()
fig.write_image("../output/eval/images/data_size_provenance_level.png", scale=2)

In [10]:
# per_doc_data = per_doc_data.loc[per_doc_data.component.isin(country_order)]

agg = per_doc_data.groupby(['mobility_v2', 'component', 'filter_reason']).size().reset_index(name='count')
# Normalize to proportions within each feature1/feature2 pair
agg['proportion'] = agg.groupby(['mobility_v2', 'component'])['count'].transform(lambda x: x / x.sum())

In [11]:
fig = px.bar(
    agg,
    x='component',
    y='proportion',
    color='filter_reason',
    facet_col='mobility_v2',
    facet_col_wrap=2,          # 2x2 grid of facets instead of 1x4 row
    # title='Category Breakdown by Feature 1 × Feature 2',
    color_discrete_sequence=px.colors.qualitative.Dark24,
    labels={
        "mobility_v2": "Register level",
        "filter_reason": "Filter reason",
        "component": "Provenance (country)",
        "proportion": "Proportion"
    }
)
fig.update_layout(barmode='stack', height=800, width=1000, font=dict(size=12))
fig.update_xaxes(tickangle=45, type="category", matches=None)
# fig.show()
fig.write_image("../output/eval/images/filter_reason_breakdown.png", scale=2)

# Rejection Rate

In [12]:
df = pmi_data["by_source_mobility_v2_cluster_shared_component"]
df["source"], df["level"], df["cluster"], df["country"] = zip(*df["value_parts"].to_list())
df = df.drop(columns=["color_key"])
df["f_level"] = 0
df.loc[df.source == "ice", "f_level"] = df.loc[df.source == "ice", "level"].map(f_level_map)
df.loc[df.f_level == 0, "level"] = "unknown (web)"
pivot_df = df.pivot(index="country", columns="level", values="cumulative_removal")
pivot_df = pivot_df.reindex(index=country_order, columns=f_order)

In [13]:
fig = px.imshow(
    pivot_df,
    labels=dict(x="Register level", y="Provenance (country)"),
    x=pivot_df.columns,
    y=[label.upper() for label in pivot_df.index],
    color_continuous_scale="Viridis",
    zmin=0, zmax=1,
    aspect="auto",
)

text_matrix = pivot_df.round(2).astype(str).values
text_matrix[pd.isna(pivot_df.values)] = "N/A"

fig.update_traces(text=text_matrix, texttemplate="%{text}")

prov_boundaries = [1.5, 5.5, 9.5]

for y_pos in prov_boundaries:
    fig.add_shape(
        type='line',
        x0=-0.5, x1=len(pivot_df.columns) - 0.5,  # spans full width of the heatmap
        y0=y_pos, y1=y_pos,
        line=dict(color='black', width=2),
        xref='x', yref='y'
    )
    
fig.update_layout(
    font=dict(size=12), 
    width=500, height=500,
    plot_bgcolor='rgba(200, 200, 200, 1)',
    coloraxis_colorbar=dict(
        orientation='h',
        title=dict(
            text='Percent documents rejected',
            side='top'          # ensures label sits above the bar, not beside it
        ),
        x=0.5,
        xanchor='center',
        y=1,                  # bar's vertical position — higher = more separation from plot
        yanchor='bottom',
        len=0.6,
        thickness=15
    ),
    margin=dict(t=100),         # generous top margin so label + bar both fit above the heatmap
    title=None   
)
# fig.show()

fig.write_image("../output/eval/images/retention/reject_comp_level.png", scale=2)

In [14]:
ab_ors = pd.read_csv("../joint_ors.csv")
ab_ors["level"] = ab_ors["level"].astype(str)
ab_ors["value_parts"] = ab_ors.level.str.split("/")
spec_feature_names = ["source", "mobility", "cluster"]
# ab_ors["color_key"] = ab_ors["value_parts"].apply(lambda parts: get_color_key(parts, spec_feature_names))
ab_ors['source'], ab_ors['level'], ab_ors['cluster'] = zip(*ab_ors['value_parts'].tolist())
df2 = ab_ors.drop(columns=["variable", "odds_ratio"])
df2.head()

,level,n,retention_p_hat,retention_low,retention_high,rejection_p_hat,rejection_low,rejection_high,value_parts,source,cluster
0,unknown (web),654544,0.084286,0.000671,0.000675,0.915714,0.000675,0.000671,"[glowbe, unknown (web), 1]",glowbe,1
1,unknown (web),448600,0.100782,0.000878,0.000884,0.899218,0.000884,0.000878,"[glowbe, unknown (web), 2]",glowbe,2
2,unknown (web),463929,0.088186,0.000813,0.000819,0.911814,0.000819,0.000813,"[glowbe, unknown (web), 3]",glowbe,3
3,unknown (web),218529,0.093164,0.001212,0.001226,0.906836,0.001226,0.001212,"[glowbe, unknown (web), 4]",glowbe,4
4,high,80,0.862500,0.092171,0.058953,0.137500,0.058953,0.092171,"[ice, high, 1]",ice,1


In [15]:
fig = px.scatter(
    df2,
    x="cluster",
    y="rejection_p_hat",
    # facet_col="cluster",
    # facet_col_wrap=2,
    error_y="rejection_high",
    error_y_minus="rejection_low",
    color="level",
    # size="cumulative_removal",
    category_orders={
        "cluster": ["1", "2", "3", "4"],
        "f_level": [0, 1, 2, 3],
        "level": ["unknown (web)", "low", "medium", "high"]
    },
    labels={
        "rejection_p_hat": "Document rejection rate",
        "cluster": "Provenance group",
        "level": "Register level"
    },
    color_discrete_map=COLOR_MAP,
)
fig.update_yaxes(
    dtick=1 
)
fig.update_xaxes(
    tickangle=45,
    type="category",
    matches=None
)
# fig.update_layout(scattermode="group")
fig.update_layout(font=dict(size=12), width=500, height=400)
# fig
fig.write_image("../output/eval/images/retention/rejection_clustered_ci.png", scale=2)
# fig.write_image("../output/eval/images/retention/rejection_comps.png", scale=2)
# fig.write_image("../output/eval/images/retention/rejection_clustered.png", scale=2)

# Pointwise Mutual Information (PMI)

*Operationalized hypothesis*: PMI(document dialect cluster; document exclusion) will trend with LEU centrality/peripherality, as represented by cluster ID. 

When measuring only by cluster ID, the results are unknown. For clusters (1-4) based on GloWbe and ICE, the results are **null**: PMI for cluster IDs and source corpora are close to zero, indicating independence between document exclusion and this representation of dialect typology. However, this does not hold for Cluster 5, where we find a positive association with document exclusion. 

Moreover, there are stronger results for document genres (though the scale is compressed when normalized due to the far smaller number of written/spoken genres compared to web blog/general). Furthermore, rather than strong positive associations of exclusion and document features (cluster, source, component, genre), the results largely indicate *disassociations* of exclusion and generally central LEU features. Interestingly, Cluster 4 and Cluster 1 documents pattern together when looking at ICE documents specifically. 

In [39]:
bar_labels = {
    "npmi": "NPMI(provenance; document excluded)",
    "pmi": "PMI(provenance; document excluded)",
    "feature_value": "Provenance (country)",
    "cumulative_removal": "Rejection Rate",
    "level": "Degree of Mobility",
    "color": "Provenance Group",
    "color_key": "Register level"
}

In [84]:
def render_bar(subset_key, bar_key, labels, bar_range, filter_val=None, filter_key="color_key", color_map=COLOR_MAP, font_size=10, ticksize=0.5):
    
    df = pmi_data[subset_key]
    if filter_val:
        df = df.loc[df[filter_key] == filter_val]
        colors = df["value_parts"].apply(lambda v: v[-1])
    else:
        colors=df["color_key"]

    fig = px.bar(
        df, 
        x=bar_key, 
        y="feature_value", 
        orientation="h", 
        # text_auto=".3f",
        labels=labels,
        color=colors,
        color_discrete_map=color_map,
        category_orders={
            "color": ["1", "2", "3", "4"],
            "feature_value": ["low", "medium", "high"]
        }
    )
    
    fig.update_xaxes(
        dtick=ticksize, 
        range=bar_range,
        tickangle=45
    )
    fig.update_yaxes(
        type="category",
        title="",
        # tickmode="array",
        # tickvals=df["feature_value"],
        # ticktext=df["value_parts"].apply(lambda v: v[1].upper())
    )
    fig.update_layout(
        font=dict(size=font_size), width=500, height=250,
        # showlegend=False
    )
    return fig


In [90]:
render_bar(
    "by_mobility_v2_ice", 
    "npmi", 
    labels=bar_labels,
    bar_range=[-0.35, 0.55],
    ticksize=0.1
) .write_image("../output/eval/images/pmi/npmi_mobility_ice.png", scale=2)

In [78]:
render_bar(
    "by_mobility_v2_component_cluster_in_ice", 
    "pmi", 
    labels=bar_labels,
    bar_range=[-3.25, 0.01],
    filter_val="medium",
    ticksize=0.25
).write_image("../output/eval/images/pmi/pmi_medium_mobility_comp_ice.png", scale=2)

In [ ]:
# render_pmi_bar("by_cluster").write_image("../output/eval/images/pmi/cluster_1-4_null.png")
# render_pmi_bar("by_source_genre_english").write_image("../output/eval/images/pmi/mobility.png")
# render_pmi_bar("by_source_genre_cluster_english").write_image("../output/eval/images/pmi/mobility_cluster.png")
# render_pmi_bar("by_mobility_v2_component_ice").write_image("../output/eval/images/retention/cluster_facet_mobility_component_ice_removal.png")

## Jensen-Shannon Divergence

A measure for bias by measuring the divergence between two probability distributions from their average. For this study, we can technically compute JSD on several possible values (total documents, included documents, excluded documents), but to normalize for frequency effects, we measure the "drift" between stages measured by value-specific retention rates. 

TODO: remove from this context and compute at the token-level to make it comparable with mixing!

In [ ]:
from scipy.spatial import distance

In [ ]:
data = pmi_data["by_source_genre_component_english"]
excluded_prob = (data.joint_count / data.joint_count.sum()).to_numpy()
total_prob = (data.marginal_count / data.marginal_count.sum()).to_numpy()
included = data.marginal_count - data.joint_count
included_prob = (included / included.sum()).to_numpy()

distance.jensenshannon(total_prob, included_prob,base=2) ** 2

In [ ]:
def get_plot_height(df):
    return max(800, 100 + (25 * len(df)))

In [ ]:
def render_cr_bar(sort_key="color_key", color_map=COLOR_MAP, font_size=10):
    # df = pmi_data["by_mobility_v2_component_ice"]
    df = ab_ors.loc[ab_ors.mobility!="high"]
    fig = px.scatter(
        df, 
        x="color_key", 
        y="rejection_p_hat", 
        error_y="rejection_high",
        error_y_minus="rejection_low",
        color="color_key",
        size="mobility",
        color_discrete_map=COLOR_MAP,
        category_orders = {
            "color_key": ["1", "2", "3", "4"],
            # "mobility": ["low", "medium", "high"]
        },
    )
    fig.update_layout(scattermode="group")
    fig.update_layout(font=dict(size=font_size), width=600, height=600)
    return fig

In [ ]:
# render_cr_bar()

# (WIP) Retention Rate (RR) and Log-Retention Penalty (LRP)
*Operationalized hypothesis*: Cluster 1 will have the highest retention rate at each stage of the pipeline. 

The results are basically **negative**: all clusters except for code-switching share nearly identical relative retention rates. This is mostly due to the volume of GloWbe data obscuring the effects *within* clusters. See results for `by_cluster_in_ice` compared to `by_cluster_in_glowbe`, or `by_genre_in_cluster_*`.

Also note that the retention rate for LinCE at stage 5 of the pipeline is highly inflated but only accounts for 48 out of 68 input documents (from a set of originally 172K documents).

In [ ]:
series["by_mobility_v2_cluster"]

## Line Charts

In [18]:
def render_retention_line(subset_key, color_map=COLOR_MAP,height=600,width=600, font_size=11):
    df = series[subset_key]
    df["level"], df["cluster"] = zip(*df["value_parts"].tolist())
    df["rejection"] = 1- df["retention"]
    fig = px.line(
        df, x="stage", y="rejection", 
        line_group="cluster", color="cluster",
        markers=True,
        # facet_row="level",
        color_discrete_map=color_map,
        category_orders={
            "color_key": ["1", "2", "3", "4"],
            "level": ["low", "medium", "high"]
        },
        labels={
            "cluster": "Provenance group",
            "level": "Register level",
            "rejection": "Document rejection rate",
            "stage": "FineWeb filtering stage"
        }
    )
    fig.update_layout(width=width, height=height, font=dict(size=font_size))
    fig.update_yaxes(range=[-0.05,1.1], tickformat=".1%", title="")
    fig.update_xaxes(range=[0.9,6.1], dtick=1)
    fig.for_each_annotation(lambda a: a.update(text=f"{a.text.split("=")[-1].title()} level"))
    # fig.update_traces(opacity=0.5) 
    return fig

In [20]:
temp = series["by_mobility_v2_cluster_glowbe"]
temp["level"], temp["cluster"] = zip(*temp["value_parts"].tolist())
temp["level"] = "unknown (web)"

df3 = series["by_mobility_v2_cluster_ice"]
df3["level"], df3["cluster"] = zip(*df3["value_parts"].tolist())
df3 = pd.concat([temp, df3])
df3["rejection"] = 1 - df3["retention"]
df3["stage"] = df3["stage"] - 1
df3

,value,stage,stage_name,included,total,retention,lrp,cumulative_lrp,value_parts,color_key,level,cluster,rejection
0,high / 1,0,📖 - READER: 🐿 Jsonl,654544,654544,1.000000,-0.000000,0.000000,"[high, 1]",high,unknown (web),1,0.000000
4,high / 1,1,🔻 - FILTER: 🌍 Language ID,653929,654544,0.999060,0.000940,0.000940,"[high, 1]",high,unknown (web),1,0.000940
8,high / 1,2,🔻 - FILTER: 👯 Gopher Repetition,628320,653929,0.960838,0.039949,0.040889,"[high, 1]",high,unknown (web),1,0.039162
12,high / 1,3,🔻 - FILTER: 🥇 Gopher Quality,604832,628320,0.962618,0.038099,0.078988,"[high, 1]",high,unknown (web),1,0.037382
16,high / 1,4,🔻 - FILTER: ⛰ C4 Quality,483748,604832,0.799806,0.223387,0.302375,"[high, 1]",high,unknown (web),1,0.200194
...,...,...,...,...,...,...,...,...,...,...,...,...,...
23,medium / 4,1,🔻 - FILTER: 🌍 Language ID,1598,1600,0.998750,0.001251,0.001251,"[medium, 4]",medium,medium,4,0.001250
35,medium / 4,2,🔻 - FILTER: 👯 Gopher Repetition,1563,1598,0.978098,0.022146,0.023397,"[medium, 4]",medium,medium,4,0.021902
47,medium / 4,3,🔻 - FILTER: 🥇 Gopher Quality,1536,1563,0.982726,0.017425,0.040822,"[medium, 4]",medium,medium,4,0.017274
59,medium / 4,4,🔻 - FILTER: ⛰ C4 Quality,1314,1536,0.855469,0.156106,0.196928,"[medium, 4]",medium,medium,4,0.144531


In [ ]:
df.stage.value_counts()

In [22]:
fig = px.line(
    df3, x="stage", y="rejection", 
    line_group="cluster", color="level",
    markers=True,
    facet_row="cluster",
    color_discrete_map=COLOR_MAP,
    category_orders={
        "color_key": ["1", "2", "3", "4"],
        "level": ["low", "medium", "high"]
    },
    labels={
        "cluster": "Provenance group",
        "level": "Register level",
        "rejection": "Document rejection rate",
        "stage": "FineWeb filtering stage"
    }
)
fig.update_layout(width=600, height=800, font=dict(size=12))
fig.update_yaxes(range=[-0.05,1.1], tickformat=".1%", title="")
fig.update_xaxes(range=[-0.05,5.1], dtick=1)
fig.for_each_annotation(lambda a: a.update(text=f"Group {a.text.split("=")[-1]}"))
# fig.show()
fig.write_image("../output/eval/images/retention/progressive_rejection.png")

In [ ]:
# rr_component_written = compute_relative_rate("by_component_written", "us")
# rr_component_written
# print(rr_genre.to_latex())
rr_mobility

In [ ]:
rr_component_genre = compute_relative_rate("by_component_genre", "us / W2B")
rr_component_genre

## Filter Contributions (Waterfall Chart)
*Operationalized hypothesis*: LangID will be associated with the highest contributions to data distortion (measured by LRP).

The results are **negative**. For these data sources, the FineWeb Quality filter contributes the most to data distortion overall.

In [ ]:
def render_algo_lrp_contribution(subset_key, color_map=COLOR_MAP, width=800, height=800, font_size=11):
    df = series[subset_key]
    fig = go.Figure(go.Waterfall(
        orientation="v",
        x=df.stage_name,
        y=df.lrp,
    ))
    fig.update_yaxes(dtick=0.1, range=[-0.1,1.1])
    fig.update_layout(width=width, height=height, font=dict(size=font_size))
    return fig

render_algo_lrp_contribution("overall", height=800)
# render_lrp_contribution("overall", height=500).write_image(f"../output/eval/images/retention/lrp_by_stage.png")

# Correlation with CMI score (ABANDONED)

*Operationalized hypothesis*: Cluster 5 document CMI score (continuous \[0, 1\]) is positively correlated with document exclusion (binary 0/1).

A Cluster 5 document's CMI score approximates the degree of code-mixing. For document exclusion after the full pipeline, results are **null**. There is close to zero correlation between CMI score and document exclusion in this scenario.

For document exclusion after only the first filter (language identification), the results are **positive**. There is weak to moderate correlation between CMI score and document exclusion in this case. The average CMI score when excluded is \~0.15, \~0.06 if forwarded to the rest of the pipeline. The average CMI score for the entire LinCE corpus is \~0.17.

This does make sense because several of the deterministic filters are oriented toward 'language-agnostic' criteria measured in terms of document character ratios and repetition; LinCE documents are individual social media posts that do not follow the same orthographic norms as more formal/traditional registers, regardless of CMI score. While LinCE documents are generally shorter (see cluster averages below), token count does not correlate with document exclusion.

In [ ]:
# Copied directly from eval.correlation console output
cmi_correlation = {
    "pipeline": {
        "corr": 0.0053,
        "p_value": 0.0268
    },
    "lang_id": {
        "corr": 0.3041,
        "p_value": 0.0
    }
}

token_correlation = {
    "pipeline": {
        "corr": -0.12,
        "p_value": 0.0
    },
    "lang_id": {
        "corr": 0.05,
        "p_value": 0.0
    },
}
cmi_df = pd.DataFrame.from_dict(cmi_correlation, orient='index')
token_df = pd.DataFrame.from_dict(token_correlation, orient='index')

In [ ]:
print(cmi_df.to_latex())

In [ ]:
for c, data in preprocess_manifest["steps"]["enrich"]["clusters"].items():
    print(c)
    curr_tokens = data["tokens"]
    curr_docs = data["documents_written"]
    print(f"Average tokens/doc: {curr_tokens / curr_docs}")

## (WIP) Volume retention

*Operationalized hypothesis*: Cluster 1 will retain the most data and LEU centrality/peripherality will trend with volume retention.

The results are **positive**: Cluster 1 indeed retains the most volume throughout the stages of filtering. Surprisingly, the delta between cluster volumes sharply declines after FineWeb quality filter is applied, which shows the strength of filters blindly predicated on document length or punctuation character ratios.

In [ ]:
def render_volume_line(subset_key, color_map=COLOR_MAP,font_size=20):
    df = series[subset_key].sort_values(by=["value", "stage"])
    key_label = subset_key.replace("by_", "").replace("_", " / ")
    fig = px.line(
        df, x="stage", y="included", color="color_key",
        facet_row="mobility",
        line_group="value",
        color_discrete_map=color_map, markers=True,
        labels={"color_key": "Cluster","stage": "Filtering Stage"},
        category_orders={"mobility": ["low", "medium", "high"]},
    )
    plot_size=800

    fig.update_layout(width=plot_size, height=plot_size,font=dict(size=font_size))
    # fig.for_each_annotation(lambda a: a.update(text=f"{a.text.split("=")[-1].title()} Mobility"))

    fig.update_yaxes(matches=None,rangemode="tozero", title_text="")
    fig.update_xaxes(range=[0.9,6.1],dtick=1)
    return fig

In [ ]:
!mkdir -p ../output/eval/images/retention
render_volume_line("by_source_genre_cluster_english").write_image("../output/eval/images/retention/stage_cluster_loss.png")

In [ ]:
# !mkdir -p ../output/eval/images/retention/volume/logscale

In [ ]:
# for key in series:
    # render_volume_line(key).show()
    # render_volume_line(key).write_image(f"../output/eval/images/retention/volume/logscale/{key}.png")

In [ ]:
# series["by_mobility"]
pmi_data["by_mobility"]

## Stacked Bar Chart (ABANDONED)

*Operationalized hypothesis:* Cluster 1 will be the least distorted by filters, measured by the lowest cumulative LRP.

The results are **negative** - Cluster 1's LRP is more or less 

In [ ]:
def render_lrp_stack(subset_key, sort_key="color_key", font_size=18):
    df = series[subset_key].sort_values(by=sort_key)
    fig = px.bar(
        df, x="value", y="lrp", color="stage_name",
        facet_row="color_key",
        labels={"lrp": "Log Rejection Penalty", "value": "Dialect Typology"}
    )
    fig.update_xaxes(matches=None)
    fig.update_yaxes(title_text="")
    fig.update_layout(
        width=800, height=800,
        font=dict(size=font_size), 
        legend=dict(
            title="Algorithmic Filter",
            # maxheight=0.1
            yanchor="middle",
            y=0.5
        )
    )
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    return fig

In [ ]:
# for key in series:
#     render_lrp_contribution(key).write_image(f"../output/eval/images/retention/contribution/{key}.png")
# !mkdir -p ../output/eval/images/lrp
# render_lrp_stack("by_cluster", "value").write_image(f"../output/eval/images/lrp/typology_cluster.png")
df = series["by_source_genre_cluster_english"]
df.loc[df.color_key=="1"]

In [ ]:
# series["by_cluster"]
render_lrp_stack("by_source_genre_cluster_english", "value")

## Render Sankey diagrams (ABANDONED)

In [ ]:
def render_sankey(subset_key, color_map=COLOR_MAP, width=1200, height=600, font_size=11):
    sd = sankeys[subset_key]
    node_colors = [
        EXCLUDED_COLOR if label == "" else color_map.get(value, "#999999")
        for label, value in zip(sd["node"]["label"], sd["node"]["customdata"])
    ]
    
    fig = go.Figure(go.Sankey(
        arrangement="fixed",
        node={**sd["node"], "color": node_colors},
        link=sd["link"]
    ))
    
    fig.update_layout(title=subset_key, width=width, height=height, font=dict(size=font_size))
    return fig

In [ ]:
render_sankey("by_mobility_v2_ice")